# What is your Erdős number?

Breadth-first search on a co-authorship graph gives every author's distance to
Erdős, and a proof that it is the shortest.

In [1]:
coauthors = {
    "Erdős":  ["Graham", "Chung"],
    "Graham": ["Erdős", "Chung", "Knuth"],
    "Chung":  ["Erdős", "Graham", "Lovász"],
    "Knuth":  ["Graham", "Garcia"],
    "Lovász": ["Chung", "Garcia"],
    "Garcia": ["Knuth", "Lovász", "You"],
    "You":    ["Garcia"],
}

In [2]:
from collections import deque

def bfs(graph, start):
    dist, parent = {start: 0}, {start: None}
    queue = deque([start])
    while queue:
        u = queue.popleft()                 # oldest discovered first: whole layers finish in order
        for v in graph[u]:
            if v not in dist:               # first discovery is the shortest
                dist[v] = dist[u] + 1
                parent[v] = u
                queue.append(v)
    return dist, parent

dist, parent = bfs(coauthors, "Erdős")
for author, d in sorted(dist.items(), key=lambda kv: kv[1]):
    print(f"{author:8} {d}")

Erdős    0
Graham   1
Chung    1
Knuth    2
Lovász   2
Garcia   3
You      4


## The chain itself

Follow the parent pointers back from any author.

In [3]:
def path_to(parent, target):
    chain = []
    while target is not None:
        chain.append(target)
        target = parent[target]
    return chain[::-1]

print(" -> ".join(path_to(parent, "You")))

Erdős -> Graham -> Knuth -> Garcia -> You


## Why the first discovery is the shortest

BFS empties layer $k$ before it starts layer $k+1$, so a vertex is first reached
at the smallest possible distance. Swap the queue for a stack and you get
depth-first search, which finds *a* chain but not the shortest one:

In [4]:
def dfs_distance(graph, start, target):
    stack, seen = [(start, 0)], set()
    while stack:
        u, d = stack.pop()                  # newest first
        if u == target:
            return d
        if u in seen:
            continue
        seen.add(u)
        for v in graph[u]:
            stack.append((v, d + 1))

print("BFS distance to You:", dist["You"], "  first path found by DFS:", dfs_distance(coauthors, "Erdős", "You"))

BFS distance to You: 4   first path found by DFS: 4


**What we used:** graphs, adjacency lists and breadth-first search
(session 5, sections 7 to 9).